# ViSTKG-QA — Chuỗi C1 → C2 → C3 (Colab GPU)

Notebook CHỈ clone repo + cài đặt + gọi script trong `src/` — không viết
logic mô hình/huấn luyện riêng ở đây. **Đã được duyệt trước cả chuỗi
C1→C2→C3 — chạy bằng "Run All", KHÔNG cần dừng xác nhận giữa các giai
đoạn.** Notebook chỉ dừng thật khi có lỗi/NaN thật (script `train.py` tự
raise RuntimeError nếu >5% mẫu 1 epoch bị NaN — xem docs/DECISIONS.md —
khi đó Colab sẽ dừng cell tại chỗ lỗi, không chạy tiếp các cell sau).

Mỗi cell IN RÕ dòng bắt đầu bằng `==RESULT==` ở cuối — copy các dòng này
dán lại cho Claude để báo cáo tiến độ.

Thứ tự: **C1 smoke test** → **C2 grid search alpha rút gọn (5 epoch/điểm, 1 seed)**
→ chọn alpha tốt nhất tự động → **C3 huấn luyện đầy đủ (alpha tốt nhất, 3 seed, tới 20 epoch, early stop patience 3)**.

## 0. Thiết lập: clone repo, cài đặt, mount Drive, xác nhận GPU

In [ ]:
# Sửa URL repo thật trước khi chạy (git remote hiện tại của dự án)
!git clone https://github.com/<TO_BE_FILLED>/stkg_vietnamese.git
%cd stkg_vietnamese

In [ ]:
# Cài đặt ban đầu từ requirements.txt (đã pin transformers==4.42.3, peft==0.12.0)
# — CELL TIẾP THEO sẽ force-reinstall + tạo stub flash_attn + XÁC NHẬN LẠI
# đúng phiên bản, vì Colab có thể đã pre-cài transformers bản mới hơn gây
# xung đột (xem docs/DECISIONS.md).
!pip install -q -r requirements.txt

## 0a. Cài đặt môi trường ĐÚNG PHIÊN BẢN (transformers/peft/flash_attn) — BẮT BUỘC

Colab mặc định cài `transformers` bản mới nhất — KHÔNG tương thích code
tùy biến của Vintern-1B-v2 (lỗi `KeyError: 'architectures'`, xác nhận
thật, xem `docs/DECISIONS.md`). Cell dưới ép đúng phiên bản đã xác nhận
hoạt động ở máy local (`transformers==4.42.3`, `peft==0.12.0`) và tạo
stub `flash_attn` rỗng (code gốc của model đã tự `try/except` fallback
naive attention khi thiếu submodule thật — stub chỉ cần qua được bước
kiểm tra tĩnh `check_imports()` của transformers, không cần cài
flash_attn thật).

**QUAN TRỌNG — chạy lại từ đầu sau MỌI lần Colab ngắt runtime:** cài đặt
package (transformers/peft/stub flash_attn) KHÔNG tồn tại qua các lần
Colab restart runtime — khác với checkpoint trên Drive (vẫn còn). Phải
"Run All" lại TOÀN BỘ notebook từ đầu (kể cả cell này), không chỉ chạy
tiếp từ giữa, mỗi khi runtime bị reset/ngắt kết nối.

In [ ]:
# 1) Pin CHÍNH XÁC transformers==4.42.3 + peft==0.12.0 — force-reinstall vì
#    Colab có sẵn version khác (mới hơn) gây xung đột đã xác nhận thật ở
#    máy local (xem docs/DECISIONS.md). --no-deps để không kéo theo việc cài
#    lại torch/các dep nặng khác không cần thiết; tokenizers cài riêng dòng dưới.
!pip install --force-reinstall --no-deps transformers==4.42.3 peft==0.12.0
!pip install "tokenizers>=0.19,<0.20"
!pip install -q -r requirements.txt

# 2) Stub flash_attn rỗng — ĐÚNG logic đã làm ở máy local (xem docs/DECISIONS.md):
#    code thật của Vintern (modeling_intern_vit.py) đã tự try/except fallback
#    naive attention khi thiếu submodule flash_attn.flash_attn_interface — chỉ
#    cần "import flash_attn" (top-level) qua được check_imports() tĩnh của
#    transformers, KHÔNG cần cài flash_attn thật (rất khó build trên Colab).
import os, site
site_packages_dirs = site.getsitepackages()
target_dir = site_packages_dirs[0] if site_packages_dirs else site.getusersitepackages()
stub_dir = os.path.join(target_dir, "flash_attn")
os.makedirs(stub_dir, exist_ok=True)
open(os.path.join(stub_dir, "__init__.py"), "w").close()
print(f"Da tao stub flash_attn rong tai: {stub_dir}")

# 3) Xác nhận đúng phiên bản TRƯỚC KHI chạy C1 — DỪNG Ở ĐÂY nếu assert lỗi
import sys, importlib
import transformers, peft
print("==RESULT== python:", sys.version.split()[0])
print("==RESULT== transformers:", transformers.__version__)
print("==RESULT== peft:", peft.__version__)
import flash_attn
print("==RESULT== flash_attn stub OK:", flash_attn.__file__)
assert transformers.__version__ == "4.42.3", f"SAI VERSION transformers: {transformers.__version__} (can 4.42.3)"
assert peft.__version__ == "0.12.0", f"SAI VERSION peft: {peft.__version__} (can 0.12.0)"
print("==RESULT== Moi truong DUNG phien ban - co the chay C1.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_ROOT = '/content/drive/MyDrive/ViSTKG-QA/checkpoints'
RESULTS_ROOT = '/content/drive/MyDrive/ViSTKG-QA/results/training'
os.makedirs(CHECKPOINT_ROOT, exist_ok=True)
os.makedirs(RESULTS_ROOT, exist_ok=True)
print(f'==RESULT== Drive mounted, checkpoint root = {CHECKPOINT_ROOT}, results root = {RESULTS_ROOT}')

In [ ]:
# Xác nhận GPU thật đang dùng — DỪNG Ở ĐÂY nếu không thấy GPU (Runtime > Change runtime type > GPU)
!nvidia-smi

## C1 — Smoke test 50 mẫu thật (ĐÃ DUYỆT TRƯỚC)

Đã PASS trên CPU local (50/50 step, 0 NaN, loss giảm 30.70→27.58, xem
docs/DECISIONS.md) — chạy lại ở đây trên GPU thật để xác nhận tốc độ +
không NaN/OOM trên GPU. Nếu cell này lỗi/NaN, DỪNG — không chạy C2/C3.

In [ ]:
!python -m src.train.smoke_test --set train.device=cuda
print('==RESULT== C1 smoke test: xem log phía trên — PASS nếu "Hoàn tất smoke test: 50/50 step thành công, 0 step NaN/lỗi" và "GIẢM".')

## C2 — Grid search alpha {0.1, 0.3, 0.5, 0.7, 0.9} (ĐÃ DUYỆT TRƯỚC)

Phiên bản rút gọn: 5 epoch/điểm, 1 seed. Tự động chọn alpha tốt nhất theo
val MRR, ghi vào `{RESULTS_ROOT}/best_alpha.txt` (Drive) — C3 đọc file này,
KHÔNG cần anh sửa tay giữa các cell.

**Resume tự động:** mỗi alpha kiểm tra trước khi train — nếu
`{CHECKPOINT_ROOT}/alpha_grid_{a}/seed42_best.pt` VÀ
`{RESULTS_ROOT}/alpha_{a}_seed42_result.json` đã tồn tại (từ lần chạy
trước, vd runtime bị ngắt giữa chừng), script IN 'đã có kết quả, bỏ qua'
và dùng lại MRR có sẵn thay vì train lại — an toàn khi phải "Run All" lại
từ đầu sau khi Colab ngắt kết nối.

In [ ]:
ALPHAS = [0.1, 0.3, 0.5, 0.7, 0.9]
for a in ALPHAS:
    !python -m src.train.train --seeds 42 --tag alpha_{a} \n        --set retrieval.alpha={a} \n        --set optimizer.max_epochs=5 \n        --set train.device=cuda \n        --set train.checkpoint_dir={CHECKPOINT_ROOT}/alpha_grid_{a} \n        --set train.results_dir={RESULTS_ROOT}

In [ ]:
!python -m src.train.select_best_alpha --results_dir {RESULTS_ROOT}
with open(f'{RESULTS_ROOT}/best_alpha.txt') as f:
    BEST_ALPHA = f.read().strip()
print(f'==RESULT== C2 hoàn tất. BEST_ALPHA = {BEST_ALPHA} (đã ghi {RESULTS_ROOT}/best_alpha.txt, C3 sẽ tự đọc file này)')

## C3 — Huấn luyện đầy đủ, alpha tốt nhất (tự động từ C2), 3 seed (ĐÃ DUYỆT TRƯỚC)

Batch=32 (gradient accumulation), tới 20 epoch, early stop MRR patience 3.
Checkpoint + kết quả MRR tốt nhất mỗi seed lưu THẲNG vào Drive ngay khi
seed đó xong (không chờ đủ 3 seed).

**Resume theo TỪNG SEED độc lập** — đây là bước tốn thời gian nhất, rủi ro
Colab ngắt runtime giữa chừng cao nhất: nếu seed 42 đã có đủ
checkpoint+kết quả (từ lần chạy trước), script bỏ qua seed đó và chỉ train
tiếp seed 43/44 còn thiếu — không train lại từ đầu.

In [ ]:
with open(f'{RESULTS_ROOT}/best_alpha.txt') as f:
    BEST_ALPHA = f.read().strip()
print(f'Dùng BEST_ALPHA={BEST_ALPHA} từ C2')

!python -m src.train.train --seeds 42 43 44 --tag full_run \n    --set retrieval.alpha={BEST_ALPHA} \n    --set train.device=cuda \n    --set train.checkpoint_dir={CHECKPOINT_ROOT}/full_run \n    --set train.results_dir={RESULTS_ROOT}

In [ ]:
import json
with open(f'{RESULTS_ROOT}/train_results_full_run.json', encoding='utf-8') as f:
    full_results = json.load(f)
print('==RESULT== C3 hoàn tất — kết quả 3 seed:')
for r in full_results:
    print(f"  seed={r['seed']}: best_val_mrr={r['best_val_mrr']:.4f}")
mean_mrr = sum(r['best_val_mrr'] for r in full_results) / len(full_results)
print(f'==RESULT== Trung bình 3 seed: val_MRR={mean_mrr:.4f}')
print(f'==RESULT== Checkpoint đã lưu tại {CHECKPOINT_ROOT}/full_run/seed{{42,43,44}}_best.pt')
print('==RESULT== C1->C2->C3 HOÀN TẤT. Copy toàn bộ các dòng ==RESULT== phía trên dán lại cho Claude.')

## (Chưa chạy tự động) C4 — Ablation, C5 — Eval đầy đủ

Chạy riêng sau khi xem kết quả C3, KHÔNG nằm trong chuỗi tự động ở trên
(C4/C5 dùng checkpoint từ `full_run`, cần xác nhận C3 ổn trước).

In [ ]:
# !python -m src.eval.run_ablations --seed 42 --execute
# !python -m src.eval.run_full_eval --checkpoint_dir {CHECKPOINT_ROOT}/full_run